# 개별종목 조합H — LightGBM

`기본모델/04.LightGBM.ipynb`과 같은 `models.lightgbm.build_lightgbm_baseline`을 가져오고
조합H 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.lightgbm import build_lightgbm_baseline  # noqa: E402

MODEL_NAME = 'LightGBM'
MODEL_BUILDER = build_lightgbm_baseline


In [2]:
# 2. 조합H의 피처 값만 지정합니다.
import json

COMBINATION = 'H'
FEATURE_COLUMNS = (
    'dist_high_60',
    'sma_gap_20_60',
    'relative_ret_5_market',
    'rsi_14',
    'hv_regime',
    'turnover_20',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)
combination_report = report["combinations"].get(COMBINATION)
if combination_report is None:
    print("아직 실측 결과가 없습니다. 아래 공통 실행 명령으로 조합을 평가하세요.")
else:
    panel = combination_report["panel"]
    print("학습 기간:", panel["first_date"], "~", panel["last_date"])
    print("학습 행·종목:", panel["model_rows"], panel["stocks"])
    folds = pd.DataFrame(combination_report["outer_fold_results"])
    model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
    fold_columns = [
        "fold", "selected_class_weight", "train_dates", "valid_start", "valid_end",
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, fold_columns].round(4))
    metric_columns = [
        "accuracy", "training_majority_baseline_accuracy",
        "accuracy_minus_training_majority_baseline", "macro_f1", "balanced_accuracy",
        "mcc", "pr_auc_macro_ovr", "down_recall", "core_harmonic_mean",
    ]
    display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


조합H 피처: ('dist_high_60', 'sma_gap_20_60', 'relative_ret_5_market', 'rsi_14', 'hv_regime', 'turnover_20')
학습 기간: 20110127 ~ 20240822
학습 행·종목: 159900 157


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.4424,0.5012,-0.0588,0.3548,0.3609,0.0515,0.3760,0.1962,0.2948
1,2,balanced,980,20150123,20150421,0.3860,0.3978,-0.0118,0.3593,0.3672,0.0564,0.3750,0.2820,0.3363
2,3,balanced,1210,20151228,20160328,0.3556,0.3762,-0.0206,0.3408,0.3453,0.0218,0.3515,0.2859,0.3245
3,4,balanced,1439,20161202,20170228,0.4551,0.4617,-0.0067,0.3997,0.4025,0.1178,0.4113,0.2291,0.3310
4,5,balanced,1669,20171113,20180207,0.4051,0.3901,0.0151,0.3842,0.3892,0.0898,0.3912,0.3845,0.3910
5,6,balanced,1899,20181024,20190118,0.4161,0.3725,0.0436,0.4143,0.4148,0.1242,0.4306,0.4037,0.4113
6,7,balanced,2129,20190930,20191224,0.4286,0.4781,-0.0495,0.3678,0.3760,0.0741,0.3919,0.3121,0.3634
7,8,balanced,2359,20200902,20201130,0.3677,0.3476,0.0201,0.3620,0.3773,0.0658,0.3787,0.5124,0.4036
8,9,NaN,2589,20210806,20211105,0.3949,0.3914,0.0034,0.3419,0.3640,0.0518,0.3671,0.2172,0.2982
9,10,balanced,2818,20220714,20221012,0.3820,0.3454,0.0366,0.3806,0.3876,0.0823,0.3811,0.2988,0.3492


,OOS 폴드 평균
accuracy,0.3999
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0030
macro_f1,0.3717
balanced_accuracy,0.3785
mcc,0.0731
pr_auc_macro_ovr,0.3863
down_recall,0.3297
core_harmonic_mean,0.3572


재실행 명령: python scripts/run_stock_model_experiment.py
